# 02 因子分析 & IC检验
计算各因子的IC值、IC_IR，分年度、分行业检验因子有效性

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath('..'))
from src.factors import calc_ic_analysis
from src.utils import calc_ic, calc_ic_ir

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

In [ ]:
df = pd.read_parquet('../data/processed_data.parquet')
print(f'数据加载完成，维度: {df.shape}')

In [ ]:
# IC 分析
ic_df, ic_summary, ic_yearly = calc_ic_analysis(df)

print('=== IC 汇总 ===')
ic_summary

In [ ]:
# 分年度 IC
print('=== 分年度 IC ===')
ic_yearly

In [ ]:
# IC 时间序列可视化
factor_names = ic_df['factor'].unique()
fig, axes = plt.subplots(len(factor_names), 1, figsize=(14, 3*len(factor_names)), sharex=True)

for ax, factor in zip(axes, factor_names):
    sub = ic_df[ic_df['factor'] == factor].copy()
    ax.plot(sub['date'], sub['IC'], alpha=0.6, linewidth=0.5, label=factor)
    ax.axhline(y=0, color='red', linestyle='--', alpha=0.4)
    ax.axhline(y=sub['IC'].mean(), color='green', linestyle='-', alpha=0.7, label=f'mean={sub["IC"].mean():.4f}')
    ax.legend(loc='best')
    ax.set_ylabel('IC')

plt.suptitle('因子IC时间序列', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 分行业 IC 热力图
df_with_industry = df.dropna(subset=['fwd_ret_5d'] + ['factor_pe', 'factor_momentum_20', 'factor_vol_60'])

industry_ic = []
for industry in df_with_industry['industry'].unique():
    sub = df_with_industry[df_with_industry['industry'] == industry]
    for factor in ['factor_pe', 'factor_momentum_20', 'factor_vol_60']:
        ic = calc_ic(sub[factor], sub['fwd_ret_5d'])
        industry_ic.append({'industry': industry, 'factor': factor, 'IC': ic})

industry_ic_df = pd.DataFrame(industry_ic)
heat_data = industry_ic_df.pivot(index='factor', columns='industry', values='IC')

plt.figure(figsize=(10, 5))
sns.heatmap(heat_data, annot=True, cmap='RdBu_r', center=0, fmt='.3f')
plt.title('分行业IC热力图')
plt.tight_layout()
plt.show()